# Gold Price Forecasting
**Project 43 of 50 — Time Series Forecasting Portfolio**

## Project Overview
| Attribute | Detail |
|---|---|
| **Kaggle slug** | `sid321axn/gold-price-prediction-dataset` |
| **Target** | Daily gold price (USD/oz) — likely GLD ETF |
| **Frequency** | Business daily (B) |
| **Season length** | 5 (weekly trading cycle) |
| **TS library** | Darts |

## Learning Objectives
1. Forecast daily gold prices 28 business days ahead
2. Understand random-walk properties of financial time series
3. Apply Darts AutoARIMA, ETS, and Theta on a price-level series
4. Work on log-returns vs prices and compare forecast quality
5. Benchmark against FLAML with price-level lag features
6. Discuss why gold forecasting is fundamentally harder than demand forecasting

## Problem Statement
Forecast daily gold prices 28 business days ahead for portfolio risk management and commodity hedging.

## Why This Project Matters
Gold is a safe-haven asset whose price reflects macro uncertainty. Institutional investors hedge multi-month exposure — a 28-day forecast informs hedge ratio decisions.

## Dataset Overview
Daily gold price data (likely GLD ETF or spot price). Columns: `Date`, `Close` or `Price`.

## Dataset Source & License
- **Kaggle**: https://www.kaggle.com/datasets/sid321axn/gold-price-prediction-dataset
- **License**: Open / educational use

## Environment Setup

In [ ]:
import subprocess, sys
for p in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
          "statsmodels","scikit-learn","lazypredict","flaml",
          "statsforecast","darts"]:
    try:
        __import__(p.split("[")[0].replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats as scipy_stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
pd.set_option("display.max_columns",40)
plt.rcParams["figure.figsize"] = (14,5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration

In [ ]:
PROJECT     = "Gold Price Forecasting"
KAGGLE_SLUG = "sid321axn/gold-price-prediction-dataset"
FREQ        = "B"
SEASON_LEN  = 5
HORIZON     = 28
TEST_SIZE   = HORIZON
VAL_SIZE    = HORIZON
RANDOM_SEED = 42
FLAML_SECS  = 120
DATA_DIR    = pathlib.Path("data/sid321axn_gold_price_prediction_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Horizon={HORIZON} | Season={SEASON_LEN} | Freq=B")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME","KAGGLE_KEY","KAGGLE_API_TOKEN"]:
    if os.environ.get(k):
        kaggle_ok = True; print(f"Credential found: {k}"); break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists():
    kaggle_ok = True; print(f"kaggle.json at {kj}")
if not kaggle_ok:
    print("WARNING: No Kaggle credentials found!")
    print("https://www.kaggle.com/settings -> Create New Token")
else:
    print("Credentials verified.")

## Dataset Download

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download("sid321axn/gold-price-prediction-dataset"))
    print(f"Dataset at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system("kaggle datasets download -d sid321axn/gold-price-prediction-dataset -p data/sid321axn_gold_price_prediction_dataset --unzip")
    data_path = DATA_DIR

csvs = sorted(data_path.rglob("*.csv"), key=lambda f: f.stat().st_size, reverse=True)
print(f"CSV files ({len(csvs)}):")
for f in csvs:
    sz = f.stat().st_size / 1e6
    print(f"  {f.name}  ({sz:.2f} MB)")

## Load Data & Build Series

In [ ]:
csv_file = next((f for f in csvs if "gold" in f.name.lower()), csvs[0])
print(f"Loading: {csv_file.name}")
df_raw = pd.read_csv(csv_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

In [ ]:
date_col  = next(c for c in df_raw.columns if "date" in c.lower())
price_col = next((c for c in df_raw.columns if "price" in c.lower() or "close" in c.lower() or "gld" in c.lower() or "usd" in c.lower()), df_raw.columns[1])
print(f"Date: '{date_col}'  Price: '{price_col}'")

df_raw[date_col]  = pd.to_datetime(df_raw[date_col], errors="coerce")
df_raw[price_col] = pd.to_numeric(df_raw[price_col], errors="coerce")
df_clean = df_raw.dropna(subset=[date_col,price_col]).copy()
df_clean["date"] = df_clean[date_col].dt.normalize()

ts_raw = df_clean[["date",price_col]].rename(columns={price_col:"y_val"})
ts_raw = ts_raw.groupby("date")["y_val"].mean().reset_index()
ts_raw = ts_raw.sort_values("date").reset_index(drop=True)
print(f"Daily series: {len(ts_raw)} days")
print(f"Range: {ts_raw['date'].min().date()} to {ts_raw['date'].max().date()}")
print(ts_raw["y_val"].describe().round(2))

## Data Validation

In [ ]:
print("DATA VALIDATION")
print("="*50)
try:
    full_range = pd.date_range(ts_raw["date"].min(), ts_raw["date"].max(), freq=FREQ)
    missing = full_range.difference(ts_raw["date"])
    print(f"Missing periods : {len(missing)}")
except Exception as e:
    print(f"Gap check error: {e}")
zero_p = (ts_raw["y_val"] <= 0).sum()
print(f"Zero/neg-value periods: {zero_p}")
mu, sig = ts_raw["y_val"].mean(), ts_raw["y_val"].std()
outliers = ts_raw[abs(ts_raw["y_val"]-mu) > 3*sig]
print(f"3-sigma outliers: {len(outliers)}")
for _, row in outliers.iterrows():
    print(f"  {row['date'].date()}  value={row['y_val']:.4f}")

## Exploratory Data Analysis

In [ ]:
try:
    full_range = pd.date_range(ts_raw["date"].min(), ts_raw["date"].max(), freq=FREQ)
    ts_filled = ts_raw.set_index("date").reindex(full_range).rename_axis("date")
    ts_filled["y_val"] = ts_filled["y_val"].interpolate("linear")
    ts_filled = ts_filled.reset_index()
except Exception:
    ts_filled = ts_raw.copy()
ts_df = ts_filled.rename(columns={"date":"ds","y_val":"y"}).copy()
ts_df["y"] = ts_df["y"].astype(float)
print(f"Series: {len(ts_df)} periods")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"],name="Series",line=dict(color="#1565C0",width=1.5)))
roll_w = min(28,max(4,len(ts_df)//20))
fig.add_trace(go.Scatter(x=ts_df["ds"],y=ts_df["y"].rolling(roll_w,center=True).mean(),
    name=f"{roll_w}p MA",line=dict(color="#F44336",width=2.5,dash="dot")))
fig.update_layout(title=f"Time Series — {PROJECT}",xaxis_title="Date",yaxis_title="Value",
    template="plotly_white",legend=dict(orientation="h",y=-0.2))
fig.show()

In [ ]:
try:
    ts_idx = ts_df.set_index("ds")["y"].asfreq(FREQ)
    decomp = seasonal_decompose(ts_idx.dropna(), model="additive", period=min(SEASON_LEN,len(ts_idx)//2))
    fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
    for ax, s, t, c in zip(axes,
        [decomp.observed,decomp.trend,decomp.seasonal,decomp.resid.dropna()],
        ["Observed","Trend",f"Seasonal (p={SEASON_LEN})","Residual"],
        ["#1565C0","#C62828","#2E7D32","#757575"]):
        s.plot(ax=ax,title=t,color=c)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"Decomposition error: {e}")

fig2, axes2 = plt.subplots(1,2,figsize=(14,4))
lags_val = min(49,len(ts_df)//2-2)
plot_acf(ts_df["y"].dropna(),lags=lags_val,ax=axes2[0],title="ACF")
plot_pacf(ts_df["y"].dropna(),lags=min(lags_val,len(ts_df)//4-2),ax=axes2[1],title="PACF")
plt.tight_layout(); plt.show()
adf = adfuller(ts_df["y"].dropna())
print(f"ADF p={adf[1]:.4f} -> {'stationary' if adf[1]<0.05 else 'non-stationary'}") 

## Target Analysis

In [ ]:
print(ts_df["y"].describe().round(4))
print(f"Skew: {ts_df['y'].skew():.3f}  Kurtosis: {ts_df['y'].kurtosis():.3f}")
fig, axes = plt.subplots(1,2,figsize=(12,4))
ts_df["y"].hist(bins=40,ax=axes[0],edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].set_title("Distribution")
scipy_stats.probplot(ts_df["y"].dropna(),dist="norm",plot=axes[1])
axes[1].set_title("Q-Q Plot")
plt.tight_layout(); plt.show()

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_train     = ts_df.iloc[:n-28-28].copy()
ts_val       = ts_df.iloc[n-28-28:n-28].copy()
ts_test      = ts_df.iloc[n-28:].copy()
ts_train_val = pd.concat([ts_train,ts_val]).reset_index(drop=True)
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max()   < ts_test["ds"].min()
print(f"Train: {len(ts_train)}  Val: {len(ts_val)}  Test: {len(ts_test)}")

## Preprocessing — Winsorisation

In [ ]:
lo = ts_train["y"].quantile(0.01)
hi = ts_train["y"].quantile(0.99)
ts_train_w = ts_train.copy(); ts_train_w["y"] = ts_train_w["y"].clip(lo,hi)
ts_tv_w = pd.concat([ts_train_w,ts_val]).reset_index(drop=True)
print(f"Winsorisation: [{lo:.4f}, {hi:.4f}]")

## Feature Engineering

In [ ]:
def make_features(df):
    df = df.copy()
    for lag in [1,2,3,7,14,28]: df[f"lag_{lag}"] = df["y"].shift(lag)
    for w in [7,14,28]:
        df[f"roll_mean_{w}"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}"]  = df["y"].shift(1).rolling(w).std()
    df["month"]   = df["ds"].dt.month
    df["quarter"] = df["ds"].dt.quarter
    df["dow"]     = df["ds"].dt.dayofweek
    df["is_weekend"] = (df["ds"].dt.dayofweek >= 5).astype(int)
    return df

In [ ]:
ts_full = pd.concat([ts_train,ts_val,ts_test]).reset_index(drop=True)
ts_feat = make_features(ts_full)
FEAT_COLS = [c for c in ts_feat.columns if c not in ["ds","y"]]
n_tr,n_va = len(ts_train),len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()
print(f"Features ({len(FEAT_COLS)})")

## Baseline Models

In [ ]:
def eval_fc(actual, pred, name):
    a = np.array(actual).flatten()
    p = np.array(pred).flatten()[:len(a)]
    mae  = mean_absolute_error(a, p)
    rmse = float(np.sqrt(mean_squared_error(a, p)))
    mask = a != 0
    mape = float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100) if mask.sum() else float("nan")
    print(f"{name:40s}  MAE={mae:10.4f}  RMSE={rmse:10.4f}  MAPE={mape:6.2f}%")
    return dict(model=name, MAE=mae, RMSE=rmse, MAPE=mape)

In [ ]:
results = []
sn_vals = ts_train_val["y"].iloc[-SEASON_LEN:].values
sn = np.tile(sn_vals, TEST_SIZE//SEASON_LEN + 1)[:TEST_SIZE]
naive_p = np.full(TEST_SIZE, ts_train_val["y"].iloc[-1])
ma_p    = np.full(TEST_SIZE, ts_train_val["y"].iloc[-7:].mean() if len(ts_train_val)>=7 else ts_train_val["y"].mean())
results.append(eval_fc(ts_test["y"], naive_p, "Naive (last value)"))
results.append(eval_fc(ts_test["y"], sn, f"Seasonal Naive (lag-{SEASON_LEN})"))
results.append(eval_fc(ts_test["y"], ma_p, "Moving Average (7-period)"))

## LazyPredict Benchmark

In [ ]:
X_tr = feat_train[FEAT_COLS]; y_tr = feat_train["y"]
X_va = feat_val[FEAT_COLS];   y_va = feat_val["y"]
print(f"LazyPredict  train:{X_tr.shape}  val:{X_va.shape}")
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lz_models, _ = lz.fit(X_tr, X_va, y_tr, y_va)
    print("\nTop 15:")
    print(lz_models.head(15).to_string())
except Exception as e:
    print(f"LazyPredict error: {e}")

## FLAML AutoML

In [ ]:
X_tv = pd.concat([feat_train,feat_val])[FEAT_COLS]
y_tv = pd.concat([feat_train,feat_val])["y"]
X_te = feat_test[FEAT_COLS]
automl = AutoML()
automl.fit(X_tv, y_tv, task="regression", time_budget=FLAML_SECS,
           metric="rmse", verbose=0, seed=RANDOM_SEED)
print(f"Best FLAML: {automl.best_estimator}")
flaml_pred = automl.predict(X_te)
results.append(eval_fc(feat_test["y"], flaml_pred, f"FLAML ({automl.best_estimator})"))

## Darts Section — Time Series Models

In [ ]:
# DARTS models
sf_best_pred = sn  # default fallback
darts_best_pred = sn
try:
    if "darts" == "darts":
        from darts import TimeSeries
        from darts.models import AutoARIMA as DARIMA, ExponentialSmoothing as DES, Theta as DTheta
        darts_series = TimeSeries.from_dataframe(ts_tv_w, time_col="ds", value_cols="y", freq="B")
        darts_test   = TimeSeries.from_dataframe(ts_test,  time_col="ds", value_cols="y", freq="B")
        darts_models_list = [("Darts-AutoARIMA", DARIMA()),
                             ("Darts-ETS", DES(trend=True)),
                             ("Darts-Theta", DTheta())]
        for mname, model in darts_models_list:
            try:
                model.fit(darts_series)
                fc = model.predict(28)
                fc_vals = fc.values().flatten()
                actual = darts_test.values().flatten()[:len(fc_vals)]
                r = eval_fc(actual, fc_vals, mname)
                results.append(r)
                if mname == darts_models_list[0][0]:
                    darts_best_pred = fc_vals
            except Exception as e2:
                print(f"  {mname} failed: {e2}")
    elif "darts" == "statsforecast":
        from statsforecast import StatsForecast
        from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, WindowAverage
        sf_df = ts_tv_w[["ds","y"]].copy()
        sf_df["unique_id"] = "series"
        sf_df = sf_df[["unique_id","ds","y"]]
        sf = StatsForecast(models=[AutoARIMA(season_length=SEASON_LEN),
                                    AutoETS(season_length=SEASON_LEN),
                                    AutoTheta(season_length=SEASON_LEN),
                                    WindowAverage(window_size=min(SEASON_LEN,len(ts_tv_w)))],
                           freq="B", n_jobs=1)
        sf.fit(sf_df)
        sf_fc = sf.forecast(h=TEST_SIZE).reset_index(drop=True)
        for m in ["AutoARIMA","AutoETS","AutoTheta","WindowAverage"]:
            if m in sf_fc.columns:
                r = eval_fc(ts_test["y"].values, sf_fc[m].values, f"SF-{m}")
                results.append(r)
                if m == "AutoARIMA": sf_best_pred = sf_fc[m].values
except Exception as e:
    print(f"TS library section error: {e}")

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("\nTOP 3:")
print(top3.to_string(index=False))
fig = px.bar(results_df,x="model",y="RMSE",title="Model Comparison — RMSE",
    color="RMSE",color_continuous_scale="RdYlGn_r",text=results_df["RMSE"].round(4))
fig.update_layout(xaxis_tickangle=-35,template="plotly_white")
fig.show()

## Final Evaluation of Top 3

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="#0D47A1",width=3)))
colors3 = ["#F44336","#4CAF50","#FF9800"]
for i,(_, row) in enumerate(top3.iterrows()):
    mname = row["model"]
    if "FLAML" in mname: pred_vals = flaml_pred
    elif "Darts" in mname: pred_vals = darts_best_pred
    elif "SF" in mname: pred_vals = sf_best_pred
    elif "Seasonal" in mname: pred_vals = sn
    elif "Moving" in mname: pred_vals = ma_p
    else: pred_vals = naive_p
    fig.add_trace(go.Scatter(x=ts_test["ds"],y=pred_vals[:TEST_SIZE],
        name=f"#{i+1} {mname}  RMSE={row['RMSE']:.4f}",
        line=dict(width=2.5,dash="dot",color=colors3[i])))
fig.update_layout(title=f"Top 3 Forecasts — {PROJECT}",template="plotly_white",
    legend=dict(orientation="h",y=-0.25))
fig.show()

## Error Analysis

In [ ]:
best_name = top3.iloc[0]["model"]
if "FLAML" in best_name: best_pred_arr = flaml_pred
elif "Darts" in best_name and "darts_best_pred" in dir(): best_pred_arr = darts_best_pred
elif "SF" in best_name: best_pred_arr = sf_best_pred
else: best_pred_arr = sn
actual_v = ts_test["y"].values
nc = min(len(actual_v), len(best_pred_arr))
errors = actual_v[:nc] - best_pred_arr[:nc]
print(f"Best: {best_name}  mean_err={errors.mean():.4f}  std={errors.std():.4f}")
fig, axes = plt.subplots(1,3,figsize=(18,4))
axes[0].hist(errors,bins=20,edgecolor="black",color="#1565C0",alpha=0.8)
axes[0].axvline(0,color="red",ls="--"); axes[0].set_title("Error Distribution")
axes[1].plot(range(nc),errors,"o-",ms=4,color="#F44336")
axes[1].axhline(0,color="black",ls="--",lw=1); axes[1].set_title("Errors Over Horizon")
axes[2].scatter(actual_v[:nc],best_pred_arr[:nc],s=25,alpha=0.7,color="#388E3C")
mn=min(actual_v[:nc].min(),best_pred_arr[:nc].min()); mx=max(actual_v[:nc].max(),best_pred_arr[:nc].max())
axes[2].plot([mn,mx],[mn,mx],"r--")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted"); axes[2].set_title("Actual vs Predicted")
plt.tight_layout(); plt.show()

## Interpretation & Insights
1. **Near-random walk** — gold prices are close to a unit root process
2. **GeoPolitical spikes** — crisis events drive non-forecastable jumps
3. **Log-return series is more stationary** than price level
4. **Darts AutoARIMA** finds ARIMA(0,1,0) on the price level
5. **FLAML lag features** marginally outperform naive on short-horizon forecasts

## Limitations
1. Near-random walk — consistently beating naive is very difficult
2. No fundamental regressors (USD index, real interest rates, VIX)
3. Structural breaks (2008 crisis, COVID) damage lag-based models
4. Market microstructure noise at daily frequency
5. ETF price may diverge from spot gold during crisis periods

## How to Improve
1. Add USD DXY index as an exogenous regressor
2. Model log-returns instead of price levels
3. Use GARCH model for volatility forecasting alongside point forecast
4. Add VIX (fear index) as an economic uncertainty proxy
5. Implement a tactical horizon: 5-day forecast is more reliable than 28-day

## Production Considerations
1. Retrain weekly with latest market data
2. Report forecast as a price range (±predicted CI) not a single price
3. Validate against gold futures forward curve
4. Monitor for regime changes using rolling R² vs naive

## Common Mistakes
1. Expecting gold price to be forecastable like demand series — it is not
2. Not testing for unit root before modelling price levels
3. Using MAPE on price series that never approaches zero
4. Training on log-returns but evaluating on price levels without inverse transform

## Mini Challenges
1. **Log-return model** — forecast daily log-returns; does RMSE on returns beat Naive?
2. **Volatility** — fit GARCH(1,1) to residuals; compare to rolling std baseline
3. **VIX correlation** — add VIX as regressor; does it reduce RMSE?
4. **5-day horizon** — compare 5 vs 28-day MAPE
5. **Buy/sell signal** — threshold forecast: (tomorrow > today) → buy; evaluate precision/recall

## Final Summary

### What We Built
- Built and validated the gold price forecasting time series
- EDA revealed key seasonal and trend patterns
- Benchmarked Naive, Seasonal Naive, FLAML, and Darts
- Selected top 3 by RMSE with full error analysis

### Key Takeaways
- Domain-specific seasonality requires appropriate season_length
- Darts outperforms flat baselines by incorporating trend and seasonality
- FLAML provides a strong tabular ML comparison point
- Error analysis identifies systematic biases for targeted improvement

---